In [0]:
########################
#Install Dependencies
########################
%pip install openpyxl
%pip install pycountry
# dbutils.library.restartPython()

In [0]:
########################
# Run All States - Transformation Tests
# Each state runs in its own cell below, loading its own gold data
# and running the full chain up to that state (same as individual notebooks)
########################

# ---- CONFIGURE WHICH STATES TO RUN ----
# Comment out any states you want to skip
states_to_run = [
    # "paymentPending",
    # "appealSubmitted",
    # "awaitingRespondentEvidence(a)",
    # "awaitingRespondentEvidence(b)",
    # "caseUnderReview",
    # "reasonsForAppealSubmitted",
    # "listing",
    # "prepareForHearing",
    # "decision",
    # "decided(a)",
    # "ftpaSubmitted(a)",
    # "decided(b)",
    # "ftpaSubmitted(b)",
    # "ftpaDecided",
    "ended",
    # "remitted",
]

from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"

from models.test_result import TestResult
from dataclasses import asdict
import os, time as perf_time

test_results_path = "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)

# Grand totals across all states
grand_results = {}  # state -> list of TestResult
grand_perf = {}     # state -> list of perf dicts
grand_run_ids = {}  # state -> run_id

print(f"States to run: {states_to_run}")

# Maps each state to the fields it re-implements (mirrors the static
# `child_fields_to_exclude` list in the corresponding per-state notebook).
# Chain-child runners receive these so the parent's old version doesn't run.
CHILD_FIELDS_TO_EXCLUDE = {
    'paymentPending': [],
    'appealSubmitted': ['paymentStatus'],
    'awaitingRespondentEvidence(a)': [],
    'awaitingRespondentEvidence(b)': [],
    'caseUnderReview': ['isAppealReferenceNumberAvailable', 'journeyType', 'paAppealTypeAipPaymentOption'],
    'reasonsForAppealSubmitted': [
        'caseManagementCategory',
        'appealWasNotSubmittedReason',
        'addressLine2AdminJ',
        'legalRepEmail',
        'legalRepGivenName',
        'legalRepFamilyNamePaperJ',
        'legalRepCompanyPaperJ',
        'legalRepHasAddress',
        'legalRepAddressUK',
        'oocAddressLine1',
        'oocAddressLine2',
        'oocAddressLine3',
        'oocAddressLine4',
        'oocLrCountryGovUkAdminJ',
        'localAuthorityPolicy',
        'paAppealTypePaymentOption',
    ],
    'listing': ['additionalTribunalResponse'],
    'prepareForHearing': [],
    'decision': [],
    'decided(a)': ['decisionAndReasonsAvailable'],
    'ftpaSubmitted(a)': [],
    'decided(b)': [],
    'ftpaSubmitted(b)': [],
    'ftpaDecided': [],
    'ended': [],
    'remitted': [],
}


In [0]:
########################
# Load Config and Reference Data (bronze/silver - shared across all states)
########################
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

for stor in [curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage]:
    spark.conf.set(f"fs.azure.account.auth.type.{stor}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{stor}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{stor}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{stor}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{stor}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
gold_base = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"

# Load bronze/silver tables (shared)
M1_silver = spark.read.format("delta").load(f"{silver_path}/silver_appealcase_detail")
M1_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs")
M2_silver = spark.read.format("delta").load(f"{silver_path}/silver_caseapplicant_detail")
M3_silver = spark.read.format("delta").load(f"{silver_path}/silver_status_detail")
C = spark.read.format("delta").load(f"{silver_path}/silver_appealcategory_detail")
bhc = spark.read.format("delta").load(f"{bronze_path}/bronze_hearing_centres")
M3_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj")
bll = spark.read.format("delta").load(f"{bronze_path}/bronze_listing_location")

try: M2_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_caseappellant_appellant")
except: pass
try: bat = spark.read.format("delta").load(f"{bronze_path}/bronze_appealtype")
except: pass
try: bhoref = spark.read.format("delta").load(f"{bronze_path}/bronze_HORef_cleansing")
except: pass
try: H_silver = spark.read.format("delta").load(f"{silver_path}/silver_history_detail")
except: pass
try: H_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history")
except: pass
try: M4_silver = spark.read.format("delta").load(f"{silver_path}/silver_transaction_detail")
except: pass
try: M4_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_transaction_transactiontype")
except: pass
try: M6_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history")
except: pass
try: bac = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcategory")
except: pass
try: docsr = spark.read.format("delta").load(f"{bronze_path}/bronze_documentsreceived")
except: pass

# Missing fields schema for gold data join
from pyspark.sql.types import StructType, StructField, StringType
missing_schema = StructType([
    StructField("appealReferenceNumber", StringType(), True),
    StructField("witness1InterpreterSignLanguage", StringType(), True),
    StructField("witness2InterpreterSignLanguage", StringType(), True),
    StructField("witness3InterpreterSignLanguage", StringType(), True),
    StructField("witness4InterpreterSignLanguage", StringType(), True),
    StructField("witness5InterpreterSignLanguage", StringType(), True),
    StructField("witness6InterpreterSignLanguage", StringType(), True),
    StructField("witness7InterpreterSignLanguage", StringType(), True),
    StructField("witness8InterpreterSignLanguage", StringType(), True),
    StructField("witness9InterpreterSignLanguage", StringType(), True),
    StructField("witness10InterpreterSignLanguage", StringType(), True),
    StructField("witness1InterpreterSpokenLanguage", StringType(), True),
    StructField("witness2InterpreterSpokenLanguage", StringType(), True),
    StructField("witness3InterpreterSpokenLanguage", StringType(), True),
    StructField("witness4InterpreterSpokenLanguage", StringType(), True),
    StructField("witness5InterpreterSpokenLanguage", StringType(), True),
    StructField("witness6InterpreterSpokenLanguage", StringType(), True),
    StructField("witness7InterpreterSpokenLanguage", StringType(), True),
    StructField("witness8InterpreterSpokenLanguage", StringType(), True),
    StructField("witness9InterpreterSpokenLanguage", StringType(), True),
    StructField("witness10InterpreterSpokenLanguage", StringType(), True),
])

def load_gold_data(state_name):
    """Load gold JSON data for a given state, with missing fields join."""
    gold_path = f"{gold_base}{state_name}"
    json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
    json_path = f"{gold_path}/{json_location.name}/JSON/"
    jd = spark.read.format("json").load(json_path)
    missing = spark.read.schema(missing_schema).json(json_path)
    jd = jd.join(missing, jd["appealReferenceNumber"] == missing["appealReferenceNumber"], "left").drop(missing["appealReferenceNumber"])
    return jd

print(f"Reference data loaded. Gold loader ready.")

In [0]:
########################
# Import all test runners
########################
from Test_Functions.paymentPending_appealType_caseData_hearingCentre_flagLabels_general import run as run_pp_chunk1
from Test_Functions.paymentPending_appellantDetails_legalRepDetails_sponsorDetails import run as run_pp_chunk2
from Test_Functions.paymentPending_partyIds_payment_remission_homeOffice_defaultMappings import run as run_pp_chunk3
from Test_Functions.paymentPending_Detained import run as run_pp_chunk4
from Test_Functions.test_helpers import classify_all
from Test_Functions.run_appealSubmitted import run_all_tests as run_as
from Test_Functions.run_awaitingRespEvidenceA import run_all_tests as run_areA
from Test_Functions.run_awaitingRespEvidenceB import run_all_tests as run_areB
from Test_Functions.run_caseUnderReview import run_all_tests as run_cur
from Test_Functions.run_reasonsForAppeal import run_all_tests as run_rfa
from Test_Functions.run_listing import run_all_tests as run_listing
from Test_Functions.run_prepareForHearing import run_all_tests as run_pfh
from Test_Functions.run_decision import run_all_tests as run_decision
from Test_Functions.run_decidedA import run_all_tests as run_decidedA
from Test_Functions.run_ftpaSubmittedA import run_all_tests as run_ftpaA
from Test_Functions.run_decidedB import run_all_tests as run_decidedB
from Test_Functions.run_ftpaSubmittedB import run_all_tests as run_ftpaB
from Test_Functions.run_ftpaDecided import run_all_tests as run_ftpaDecided
from Test_Functions.run_ended import run_all_tests as run_ended
from Test_Functions.run_remitted import run_all_tests as run_remitted

########################
# PROCESS RESULTS
########################
from collections import defaultdict
from Test_Functions.report_helpers import (
    count_by_status,
    create_run, create_results, create_perf_results,
    build_report_folder,
    write_csv, write_xlsx, write_html,
)

print("All test runners imported.")

In [0]:
########################
# Run Tests for: paymentPending
# Runs: paymentPending
########################
if "paymentPending" in states_to_run:
    current_state = "paymentPending"
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: paymentPending (4 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("paymentPending")
    print(f"  Gold data loaded for paymentPending: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/4] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/4] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/4] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/4] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for paymentPending
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="paymentPending")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "paymentPending")
        grand_run_ids["paymentPending"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for paymentPending (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "paymentPending")
        write_csv(all_test_results, "paymentPending", folder, timestamp)
        write_xlsx(all_test_results, "paymentPending", folder, timestamp)
        write_html(all_test_results, "paymentPending", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["paymentPending"] = all_test_results
    grand_perf["paymentPending"] = perf_timings
    print(f"  paymentPending COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping paymentPending")

In [0]:
########################
# Run Tests for: appealSubmitted
# Runs: paymentPending -> appealSubmitted
########################
if "appealSubmitted" in states_to_run:
    current_state = "appealSubmitted"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: appealSubmitted (5 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("appealSubmitted")
    print(f"  Gold data loaded for appealSubmitted: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/5] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/5] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/5] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/5] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/5] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for appealSubmitted
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="appealSubmitted")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "appealSubmitted")
        grand_run_ids["appealSubmitted"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for appealSubmitted (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "appealSubmitted")
        write_csv(all_test_results, "appealSubmitted", folder, timestamp)
        write_xlsx(all_test_results, "appealSubmitted", folder, timestamp)
        write_html(all_test_results, "appealSubmitted", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["appealSubmitted"] = all_test_results
    grand_perf["appealSubmitted"] = perf_timings
    print(f"  appealSubmitted COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping appealSubmitted")

In [0]:
########################
# Run Tests for: awaitingRespondentEvidence(a)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a)
########################
if "awaitingRespondentEvidence(a)" in states_to_run:
    current_state = "awaitingRespondentEvidence(a)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: awaitingRespondentEvidence(a) (6 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("awaitingRespondentEvidence(a)")
    print(f"  Gold data loaded for awaitingRespondentEvidence(a): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/6] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/6] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/6] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/6] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/6] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/6] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for awaitingRespondentEvidence(a)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="awaitingRespondentEvidence(a)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "awaitingRespondentEvidence(a)")
        grand_run_ids["awaitingRespondentEvidence(a)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for awaitingRespondentEvidence(a) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "awaitingRespondentEvidence(a)")
        write_csv(all_test_results, "awaitingRespondentEvidence(a)", folder, timestamp)
        write_xlsx(all_test_results, "awaitingRespondentEvidence(a)", folder, timestamp)
        write_html(all_test_results, "awaitingRespondentEvidence(a)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["awaitingRespondentEvidence(a)"] = all_test_results
    grand_perf["awaitingRespondentEvidence(a)"] = perf_timings
    print(f"  awaitingRespondentEvidence(a) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping awaitingRespondentEvidence(a)")

In [0]:
########################
# Run Tests for: awaitingRespondentEvidence(b)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b)
########################
if "awaitingRespondentEvidence(b)" in states_to_run:
    current_state = "awaitingRespondentEvidence(b)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: awaitingRespondentEvidence(b) (7 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("awaitingRespondentEvidence(b)")
    print(f"  Gold data loaded for awaitingRespondentEvidence(b): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/7] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/7] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/7] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/7] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/7] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/7] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/7] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for awaitingRespondentEvidence(b)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="awaitingRespondentEvidence(b)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "awaitingRespondentEvidence(b)")
        grand_run_ids["awaitingRespondentEvidence(b)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for awaitingRespondentEvidence(b) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "awaitingRespondentEvidence(b)")
        write_csv(all_test_results, "awaitingRespondentEvidence(b)", folder, timestamp)
        write_xlsx(all_test_results, "awaitingRespondentEvidence(b)", folder, timestamp)
        write_html(all_test_results, "awaitingRespondentEvidence(b)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["awaitingRespondentEvidence(b)"] = all_test_results
    grand_perf["awaitingRespondentEvidence(b)"] = perf_timings
    print(f"  awaitingRespondentEvidence(b) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping awaitingRespondentEvidence(b)")

In [0]:
########################
# Run Tests for: caseUnderReview
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview
########################
if "caseUnderReview" in states_to_run:
    current_state = "caseUnderReview"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: caseUnderReview (8 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("caseUnderReview")
    print(f"  Gold data loaded for caseUnderReview: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/8] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/8] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/8] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/8] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/8] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/8] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/8] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/8] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for caseUnderReview
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="caseUnderReview")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "caseUnderReview")
        grand_run_ids["caseUnderReview"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for caseUnderReview (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "caseUnderReview")
        write_csv(all_test_results, "caseUnderReview", folder, timestamp)
        write_xlsx(all_test_results, "caseUnderReview", folder, timestamp)
        write_html(all_test_results, "caseUnderReview", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["caseUnderReview"] = all_test_results
    grand_perf["caseUnderReview"] = perf_timings
    print(f"  caseUnderReview COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping caseUnderReview")

In [0]:
########################
# Run Tests for: reasonsForAppealSubmitted
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted
########################
if "reasonsForAppealSubmitted" in states_to_run:
    current_state = "reasonsForAppealSubmitted"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: reasonsForAppealSubmitted (9 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("reasonsForAppealSubmitted")
    print(f"  Gold data loaded for reasonsForAppealSubmitted: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/9] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/9] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/9] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/9] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/9] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/9] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/9] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/9] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/9] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for reasonsForAppealSubmitted
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="reasonsForAppealSubmitted")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "reasonsForAppealSubmitted")
        grand_run_ids["reasonsForAppealSubmitted"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for reasonsForAppealSubmitted (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "reasonsForAppealSubmitted")
        write_csv(all_test_results, "reasonsForAppealSubmitted", folder, timestamp)
        write_xlsx(all_test_results, "reasonsForAppealSubmitted", folder, timestamp)
        write_html(all_test_results, "reasonsForAppealSubmitted", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["reasonsForAppealSubmitted"] = all_test_results
    grand_perf["reasonsForAppealSubmitted"] = perf_timings
    print(f"  reasonsForAppealSubmitted COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping reasonsForAppealSubmitted")

In [0]:
########################
# Run Tests for: listing
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing
########################
if "listing" in states_to_run:
    current_state = "listing"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: listing (10 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("listing")
    print(f"  Gold data loaded for listing: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/10] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/10] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/10] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/10] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/10] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/10] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/10] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/10] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/10] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/10] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for listing
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="listing")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "listing")
        grand_run_ids["listing"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for listing (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "listing")
        write_csv(all_test_results, "listing", folder, timestamp)
        write_xlsx(all_test_results, "listing", folder, timestamp)
        write_html(all_test_results, "listing", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["listing"] = all_test_results
    grand_perf["listing"] = perf_timings
    print(f"  listing COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping listing")

In [0]:
########################
# Run Tests for: prepareForHearing
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing
########################
if "prepareForHearing" in states_to_run:
    current_state = "prepareForHearing"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: prepareForHearing (11 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("prepareForHearing")
    print(f"  Gold data loaded for prepareForHearing: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/11] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/11] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/11] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/11] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/11] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/11] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/11] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/11] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/11] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/11] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/11] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for prepareForHearing
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="prepareForHearing")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "prepareForHearing")
        grand_run_ids["prepareForHearing"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for prepareForHearing (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "prepareForHearing")
        write_csv(all_test_results, "prepareForHearing", folder, timestamp)
        write_xlsx(all_test_results, "prepareForHearing", folder, timestamp)
        write_html(all_test_results, "prepareForHearing", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["prepareForHearing"] = all_test_results
    grand_perf["prepareForHearing"] = perf_timings
    print(f"  prepareForHearing COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping prepareForHearing")

In [0]:
########################
# Run Tests for: decision
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision
########################
if "decision" in states_to_run:
    current_state = "decision"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: decision (12 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("decision")
    print(f"  Gold data loaded for decision: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/12] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/12] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/12] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/12] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/12] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/12] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/12] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/12] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/12] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/12] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/12] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/12] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for decision
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="decision")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "decision")
        grand_run_ids["decision"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for decision (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "decision")
        write_csv(all_test_results, "decision", folder, timestamp)
        write_xlsx(all_test_results, "decision", folder, timestamp)
        write_html(all_test_results, "decision", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["decision"] = all_test_results
    grand_perf["decision"] = perf_timings
    print(f"  decision COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping decision")

In [0]:
########################
# Run Tests for: decided(a)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a)
########################
if "decided(a)" in states_to_run:
    current_state = "decided(a)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: decided(a) (13 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("decided(a)")
    print(f"  Gold data loaded for decided(a): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/13] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/13] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/13] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/13] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/13] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/13] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/13] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/13] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/13] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/13] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/13] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/13] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/13] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for decided(a)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="decided(a)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "decided(a)")
        grand_run_ids["decided(a)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for decided(a) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "decided(a)")
        write_csv(all_test_results, "decided(a)", folder, timestamp)
        write_xlsx(all_test_results, "decided(a)", folder, timestamp)
        write_html(all_test_results, "decided(a)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["decided(a)"] = all_test_results
    grand_perf["decided(a)"] = perf_timings
    print(f"  decided(a) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping decided(a)")

In [0]:
########################
# Run Tests for: ftpaSubmitted(a)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a)
########################
if "ftpaSubmitted(a)" in states_to_run:
    current_state = "ftpaSubmitted(a)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: ftpaSubmitted(a) (14 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("ftpaSubmitted(a)")
    print(f"  Gold data loaded for ftpaSubmitted(a): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/14] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/14] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/14] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/14] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/14] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/14] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/14] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/14] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/14] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/14] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/14] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/14] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/14] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [14/14] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for ftpaSubmitted(a)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="ftpaSubmitted(a)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "ftpaSubmitted(a)")
        grand_run_ids["ftpaSubmitted(a)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for ftpaSubmitted(a) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "ftpaSubmitted(a)")
        write_csv(all_test_results, "ftpaSubmitted(a)", folder, timestamp)
        write_xlsx(all_test_results, "ftpaSubmitted(a)", folder, timestamp)
        write_html(all_test_results, "ftpaSubmitted(a)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["ftpaSubmitted(a)"] = all_test_results
    grand_perf["ftpaSubmitted(a)"] = perf_timings
    print(f"  ftpaSubmitted(a) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping ftpaSubmitted(a)")

In [0]:
########################
# Run Tests for: decided(b)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a) -> decided(b)
########################
if "decided(b)" in states_to_run:
    current_state = "decided(b)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: decided(b) (15 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("decided(b)")
    print(f"  Gold data loaded for decided(b): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/15] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/15] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/15] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/15] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/15] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/15] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/15] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/15] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/15] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/15] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/15] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/15] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/15] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [14/15] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [15/15] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for decided(b)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="decided(b)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "decided(b)")
        grand_run_ids["decided(b)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for decided(b) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "decided(b)")
        write_csv(all_test_results, "decided(b)", folder, timestamp)
        write_xlsx(all_test_results, "decided(b)", folder, timestamp)
        write_html(all_test_results, "decided(b)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["decided(b)"] = all_test_results
    grand_perf["decided(b)"] = perf_timings
    print(f"  decided(b) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping decided(b)")

In [0]:
########################
# Run Tests for: ftpaSubmitted(b)
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a) -> decided(b) -> ftpaSubmitted(b)
########################
if "ftpaSubmitted(b)" in states_to_run:
    current_state = "ftpaSubmitted(b)"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: ftpaSubmitted(b) (16 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("ftpaSubmitted(b)")
    print(f"  Gold data loaded for ftpaSubmitted(b): {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/16] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/16] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/16] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/16] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/16] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/16] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/16] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/16] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/16] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/16] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/16] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/16] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/16] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [14/16] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [15/16] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(b)", "test_from_state": "ftpaSubmitted(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [16/16] ftpaSubmitted(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for ftpaSubmitted(b)
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="ftpaSubmitted(b)")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "ftpaSubmitted(b)")
        grand_run_ids["ftpaSubmitted(b)"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for ftpaSubmitted(b) (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "ftpaSubmitted(b)")
        write_csv(all_test_results, "ftpaSubmitted(b)", folder, timestamp)
        write_xlsx(all_test_results, "ftpaSubmitted(b)", folder, timestamp)
        write_html(all_test_results, "ftpaSubmitted(b)", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["ftpaSubmitted(b)"] = all_test_results
    grand_perf["ftpaSubmitted(b)"] = perf_timings
    print(f"  ftpaSubmitted(b) COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping ftpaSubmitted(b)")

In [0]:
########################
# Run Tests for: ftpaDecided
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a) -> decided(b) -> ftpaSubmitted(b) -> ftpaDecided
########################
if "ftpaDecided" in states_to_run:
    current_state = "ftpaDecided"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: ftpaDecided (17 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("ftpaDecided")
    print(f"  Gold data loaded for ftpaDecided: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/17] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/17] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/17] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/17] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/17] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/17] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/17] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/17] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/17] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/17] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/17] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/17] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/17] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [14/17] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [15/17] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(b)", "test_from_state": "ftpaSubmitted(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [16/17] ftpaSubmitted(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaDecided...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaDecided(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaDecided", "test_from_state": "ftpaDecided", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [17/17] ftpaDecided".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for ftpaDecided
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="ftpaDecided")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "ftpaDecided")
        grand_run_ids["ftpaDecided"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for ftpaDecided (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "ftpaDecided")
        write_csv(all_test_results, "ftpaDecided", folder, timestamp)
        write_xlsx(all_test_results, "ftpaDecided", folder, timestamp)
        write_html(all_test_results, "ftpaDecided", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["ftpaDecided"] = all_test_results
    grand_perf["ftpaDecided"] = perf_timings
    print(f"  ftpaDecided COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping ftpaDecided")

In [0]:
########################
# Run Tests for: ended
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a) -> decided(b) -> ftpaSubmitted(b) -> ftpaDecided -> ended
########################
if "ended" in states_to_run:
    current_state = "ended"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: ended (18 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("ended")
    print(f"  Gold data loaded for ended: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    # print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    # t0 = perf_time.time()
    # res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    # all_test_results.extend(res)
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [1/18] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    # t0 = perf_time.time()
    # res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    # all_test_results.extend(res)
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [2/18] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    # t0 = perf_time.time()
    # res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    # all_test_results.extend(res)
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [3/18] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting paymentPending Part 4 (detained)...")
    # t0 = perf_time.time()
    # res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    # all_test_results.extend(res)
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [4/18] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    # print(f"  Starting appealSubmitted...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [5/18] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting awaitingRespondentEvidence(a)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [6/18] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting awaitingRespondentEvidence(b)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [7/18] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting caseUnderReview...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [8/18] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting reasonsForAppealSubmitted...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [9/18] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting listing...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [10/18] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting prepareForHearing...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [11/18] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting decision...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [12/18] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting decided(a)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [13/18] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting ftpaSubmitted(a)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [14/18] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting decided(b)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [15/18] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting ftpaSubmitted(b)...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_ftpaB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "ftpaSubmitted(b)", "test_from_state": "ftpaSubmitted(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [16/18] ftpaSubmitted(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    # print(f"  Starting ftpaDecided...")
    # t0 = perf_time.time()
    # all_test_results.extend(run_ftpaDecided(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    # dur = perf_time.time() - t0
    # perf_timings.append({"test_name": "ftpaDecided", "test_from_state": "ftpaDecided", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    # print(f"  [17/18] ftpaDecided".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ended...")
    t0 = perf_time.time()
    all_test_results.extend(run_ended(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ended", "test_from_state": "ended", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [18/18] ended".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for ended
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="ended")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "ended")
        grand_run_ids["ended"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for ended (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "ended")
        write_csv(all_test_results, "ended", folder, timestamp)
        write_xlsx(all_test_results, "ended", folder, timestamp)
        write_html(all_test_results, "ended", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["ended"] = all_test_results
    grand_perf["ended"] = perf_timings
    print(f"  ended COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping ended")

In [0]:
########################
# Run Tests for: remitted
# Runs: paymentPending -> appealSubmitted -> awaitingRespondentEvidence(a) -> awaitingRespondentEvidence(b) -> caseUnderReview -> reasonsForAppealSubmitted -> listing -> prepareForHearing -> decision -> decided(a) -> ftpaSubmitted(a) -> decided(b) -> ftpaSubmitted(b) -> ftpaDecided -> ended -> remitted
########################
if "remitted" in states_to_run:
    current_state = "remitted"
    child_fields_to_exclude = CHILD_FIELDS_TO_EXCLUDE.get(current_state, [])
    print(f"\n{'=' * 80}")
    print(f"RUNNING STATE: remitted (19 steps)")
    print(f"{'=' * 80}")
    run_start_datetime = datetime.now()
    json_data = load_gold_data("remitted")
    print(f"  Gold data loaded for remitted: {json_data.count()} records")
    all_test_results = []
    perf_timings = []

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [1/19] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [2/19] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [3/19] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, current_state)
    all_test_results.extend(res)
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [4/19] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/19] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/19] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/19] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting caseUnderReview...")
    t0 = perf_time.time()
    all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/19] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting reasonsForAppealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/19] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/19] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/19] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [12/19] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [13/19] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [14/19] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [15/19] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaSubmitted(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaSubmitted(b)", "test_from_state": "ftpaSubmitted(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [16/19] ftpaSubmitted(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ftpaDecided...")
    t0 = perf_time.time()
    all_test_results.extend(run_ftpaDecided(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ftpaDecided", "test_from_state": "ftpaDecided", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [17/19] ftpaDecided".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting ended...")
    t0 = perf_time.time()
    all_test_results.extend(run_ended(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "ended", "test_from_state": "ended", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [18/19] ended".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting remitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_remitted(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "remitted", "test_from_state": "remitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [19/19] remitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    all_test_results = classify_all(all_test_results)
    # Write results for remitted
    counts = count_by_status(all_test_results)
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(spark, run_user=run_user, run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name, run_tag=run_tag,
            run_status=run_status, state_under_test="remitted")
        n = create_results(spark, run_id, all_test_results)
        np = create_perf_results(spark, run_id, perf_timings, "remitted")
        grand_run_ids["remitted"] = run_id
        print(f"  Saved run_id={run_id} ({n} results, {np} perf rows)")
    except Exception as e:
        print(f"  Spark write failed: {e}")
    # Write report files for remitted (CSV / XLSX / HTML)
    try:
        folder, timestamp, _ = build_report_folder(test_results_path, "remitted")
        write_csv(all_test_results, "remitted", folder, timestamp)
        write_xlsx(all_test_results, "remitted", folder, timestamp)
        write_html(all_test_results, "remitted", folder, timestamp, counts=counts)
        print(f"  Reports written to {folder}")
    except Exception as e:
        print(f"  Report write failed: {e}")
    grand_results["remitted"] = all_test_results
    grand_perf["remitted"] = perf_timings
    print(f"  remitted COMPLETE: {len(all_test_results)} results ({counts['PASS']} pass, {counts['FAIL']} fail, {counts['ERROR']} error, {counts['NO_DATA']} no_data)")
else:
    print(f"\n  Skipping remitted")

In [0]:
########################
# Grand Summary
########################
print(f"\n{'=' * 80}")
print(f"ALL STATES COMPLETE")
print(f"{'=' * 80}")
for s in states_to_run:
    if s in grand_results:
        c = count_by_status(grand_results[s])
        rid = grand_run_ids.get(s, "?")
        print(f"  {s:40s}  {c['TOTAL']:4d} tests  P:{c['PASS']}  F:{c['FAIL']}  E:{c['ERROR']}  ND:{c['NO_DATA']}  run_id:{str(rid)[:12]}...")
    else:
        print(f"  {s:40s}  SKIPPED")
total = sum(len(r) for r in grand_results.values())
print(f"{'=' * 80}")
print(f"  TOTAL: {total} results across {len(grand_results)} states")